# Models

Looking at the lower level API of Transformers - the models that wrap PyTorch code for the transformers themselves.

### Bitsandbytes:
bitsandbytes is a lightweight Python wrapper for CUDA custom functions, primarily used to optimize large language models (LLMs) by reducing memory usage through 8-bit and 4-bit quantization. It enables efficient inference (LLM.int8()) and training (QLoRA) on limited hardware, supporting NVIDIA CUDA 11.8 - 13.0 and offering experimental AMD GPU support.

Key Features & Capabilities:
- 4-bit and 8-bit Quantization: Enables running large models (like LLaMA, Mistral) on smaller GPUs without significant performance loss.
- QLoRA (Quantized LoRA): Allows fine-tuning of 65B+ parameter models on a single GPU by loading base models in 4-bit.
- 8-bit Optimizers: Reduces memory footprint for optimization states (e.g., Adam Optimizer), freeing up memory for larger batch sizes.
- Stable Embedding Layers: Improves stability during training with optimized embeddings.

Main Use Cases:
- Inference: Running quantized models to reduce VRAM requirements.
- Fine-tuning: Training large models using low-resource setups.
- 8-bit Matrix Multiplication: Faster and more memory-efficient computations.

### Accelerate:
Hugging Face Accelerate is an open-source library that allows PyTorch users to run their training scripts on any distributed configuration (single GPU, multi-GPU, TPU, etc.) with minimal code changes.

Key Feature:
- It abstracts the boilerplate code required for distributed training and mixed precision (FP16, BF16), leaving the core training loop unchanged.

Core Components:
- The Accelerator Class: The main object used to prepare models, optimizers, and data loaders for any environment.
- CLI Launcher: Use accelerate config to set up your environment and accelerate launch to execute scripts.

Big Model Inference: It includes utilities for loading and running extremely large models that do not fit in a single device's memory.

In [1]:
# Installing Libraries:

# Run code line below if 'bitsandbytes' and 'accelerate' libraries are not already installed:
#!pip install bitsandbytes accelerate

In [2]:
# Logging in to Hugging Face:

import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv(override= True, verbose= True)

hf_token = os.getenv("HF_TOKEN")

if hf_token and hf_token.startswith('hf_'):
    print('Hugging Face Token Found.')
    login(token= hf_token, add_to_git_credential= True)
    print('Login to Hugging Face Successful.')
else:
    print('Hugging Face Token Not Found.')

Hugging Face Token Found.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Login to Hugging Face Successful.


In [3]:
# Importing Necessary Libraries:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [4]:
# Defining Models:

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

`Note: All models defined above are intentionally choosen because of their small size to download faster and make it possible to run them on local GPU.`

In [5]:
# Defining Messages:

messages = [{'role': 'user', 'content': 'Tell a joke for a room of Data Scientists.'}]

In [6]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_quant_type= 'nf4'
)

In [7]:
# Tokenizer:

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors= 'pt').to('cuda')

print(inputs)

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2371,   5186,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116,     13, 128009]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [8]:
# The LLAMA Model:
# Note: This will download the whole LLAMA model (defined above) on local storage, so, it will take some time to run:
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map= 'auto', quantization_config= quant_config)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [9]:
# Memory used by LLAMA model:
print(f'Memory Footprint of LLAMA Model (in Bytes): {model.get_memory_footprint()}')
print(f'Memory Footprint of LLAMA Model (in KBs): {model.get_memory_footprint() / 1024}')
print(f'Memory Footprint of LLAMA Model (in MBs): {model.get_memory_footprint() / (1024*1024)}')

Memory Footprint of LLAMA Model (in Bytes): 1012011264
Memory Footprint of LLAMA Model (in KBs): 988292.25
Memory Footprint of LLAMA Model (in MBs): 965.129150390625


### Looking under the hood at the Transformer model:

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, this also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [10]:
# Execute this cell and look at what gets printed; investigate the layers

model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

## 🦙 Deconstructing the Llama-3.2-1B-Instruct Architecture
The above output is the PyTorch representation of the model. It shows exactly how data (text) flows from the moment it enters the model to the moment the model predicts the next word.

Let's break down the anatomy piece by piece.

1. The Wrapper: LlamaForCausalLM
- What it means: CausalLM stands for Causal Language Modeling. This simply means the model is trained to look at a sequence of words and predict the very next word, one step at a time (like auto-complete).
- It is split into two main parts: the model (the brain) and the lm_head (the mouthpiece).

2. The Core Brain: LlamaModel
- This is the heavy lifter. It processes the text and understands the context.

- embed_tokens: Embedding(128256, 2048)

    - Vocabulary: 128,256. The model knows 128,256 different "tokens" (words, sub-words, or characters).

    - Dimensions: 2048. When you feed it a word, it turns that word into a rich mathematical vector with 2048 numbers to represent its meaning.

- layers: (0-15)
    The model has 16 identical layers stacked on top of each other. As data passes through each layer, the model's understanding of the context gets deeper.

3. Inside a Single Layer: LlamaDecoderLayer
Every one of those 16 layers contains two main engines:

Engine A: Attention (self_attn)
This allows the model to look at other words in the sentence to understand the context (e.g., figuring out that "bank" means a river bank, not a financial bank, based on surrounding words).

   - q_proj, k_proj, v_proj: These are the Query, Key, and Value matrices used in attention.

   - Notice the sizes: The Query (q_proj) outputs 2048 dimensions, but Key and Value (k_proj, v_proj) only output 512. This tells us Llama 3 uses Grouped-Query Attention (GQA). It groups heads together to save memory and speed up inference!

   - Linear4bit: This is a massive clue! The 4bit means this model was loaded using Quantization (via bitsandbytes). It has compressed its weights to use less RAM so it can fit comfortably on your GPU.

Engine B: Feed Forward Network (mlp)
After the model pays attention to context, it passes the data through an MLP (Multi-Layer Perceptron) to "process" what it just learned.

   - gate_proj, up_proj, down_proj: It expands the data from 2048 dimensions up to 8192, mixes it, and squashes it back down to 2048.

   - act_fn: SiLUActivation(): This is the SwiGLU activation function. It determines which mathematical "neurons" should fire and which shouldn't.

The Stabilizers: LlamaRMSNorm

   You'll see input_layernorm and post_attention_layernorm. These are normalization layers that keep the numbers from exploding to infinity or shrinking to zero as they pass through the 16 layers. RMSNorm is a faster, optimized version of standard layer normalization.

4. Position Tracking: LlamaRotaryEmbedding
rotary_emb: Also known as RoPE (Rotary Position Embeddings). Unlike older models that just tag words with a "position number" at the very beginning, RoPE mathematically rotates the vectors inside the attention layer. This helps the model understand the exact distance between words (e.g., word A is exactly 3 spaces away from word B), which is crucial for logic and coding.

5. The Mouthpiece: lm_head
    - Linear(in_features=2048, out_features=128256, bias=False)

    - After passing through all 16 layers, the data is a highly complex 2048-dimensional vector.

- The lm_head takes that 2048-dimensional vector and maps it back to the original 128,256 vocabulary size.

- It outputs a probability score for every single token in its vocabulary. The token with the highest score is the word the model decides to generate next!

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not necessary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [14]:
# Running The LLAMA Model:

outputs = model.generate(inputs['input_ids'], max_new_tokens= 150)
outputs

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2371,   5186,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116,     13, 128009, 128006,  78191, 128007,    271,
             32,  22380,    369,    264,   3130,    315,    828,  14248,   1473,
          10445,   1550,    279,  10550,    733,    311,  15419,   1980,  18433,
            433,    574,   8430,    264,   2697,    330,    359,  52243,      1,
            323,   4460,    311,    330,  31690,      1,   1202,  21958,     13,
         128009]], device='cuda:0')

In [15]:
# Decoding the output using Tokenizer:
tokenizer.decode(outputs)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Apr 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a joke for a room of Data Scientists.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nA joke for a room of data scientists:\n\nWhy did the dataset go to therapy?\n\nBecause it was feeling a little "unstructured" and needed to "normalize" its emotions.<|eot_id|>']

**LOL!! Good One LLAMA!**

## A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back.
To stream results, we simply replace:
`outputs = model.generate(inputs, max_new_tokens=80)`
With:
`streamer = TextStreamer(tokenizer)`
`outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)`

Also I've added the argument `add_generation_prompt=True` to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

In [23]:
# Function to wrap everything above - and adding streaming and generation prompts:

def generate(model, messages, quant= True, max_new_tokens= 150):

    # Tokenizer:
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token

    # Input to LLM:
    inputs = tokenizer.apply_chat_template(messages, return_tensors= 'pt', add_generation_prompt= True).to('cuda')
    #attention_mask = torch.ones_like(inputs_ids, dtype=torch.long, device= 'cuda')

    # Text Streamer:
    streamer = TextStreamer(tokenizer)

    # Quantization:
    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config= quant_config).to('cuda')
    else:
        model = AutoModelForCausalLM.from_pretrained(model).to('cuda')

    # Output:
    outputs = model.generate(inputs['input_ids'], attention_mask= inputs['attention_mask'], max_new_tokens= max_new_tokens, streamer= streamer)

### PHI:

In [24]:
generate(model= PHI, messages= messages, quant= True, max_new_tokens= 150)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--microsoft--Phi-4-mini-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

<|user|>Tell a joke for a room of Data Scientists.<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why did the data scientist break up with the computer?

Because it kept asking for more space!<|end|>


**LOL!! Good one from PHI too!**

### GEMMA:

Google also now requires you to accept their terms in HuggingFace before you use Gemma.

Please visit their model page at this link and confirm you're OK with their terms, so that you're granted access.

https://huggingface.co/google/gemma-3-270m-it

In [25]:
generate(model= GEMMA, messages= messages, quant= False, max_new_tokens= 150)

config.json: 0.00B [00:00, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--google--gemma-3-270m-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<bos><start_of_turn>user
Tell a joke for a room of Data Scientists.<end_of_turn>
<start_of_turn>model
Why don't scientists ever study data? 
 
... Because they're afraid of making mistakes!
<end_of_turn>


**Not that good!! But again, it's a tiny model with just 270 Million Parameters!**

### QWEN:

In [26]:
generate(model= QWEN, messages= messages, quant= True, max_new_tokens= 150)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--Qwen--Qwen3-4B-Instruct-2507. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

<|im_start|>user
Tell a joke for a room of Data Scientists.<|im_end|>
<|im_start|>assistant
Sure! Here's a *data scientist-approved* joke with a side of statistical puns:

---

Why did the data scientist break up with their partner?

Because they found out they were *correlated* with a lot of noise and *not significantly* compatible in the long term.

Also, they kept saying, “We’re just *independent* variables in this relationship,” and it just wasn’t *p-value* worth it.

Bonus points if you laughed at the *p-value* reference and the *independent variables* line. 📊😄

---

(And if you're a true data scientist — you know that even *independent* variables can be *dependent* in real life. 😏)<|im_end|>


**Nice one from QWEN too, and a bit nerdy too. To my liking! :D**

### DEEPSEEK:

In [27]:
generate(model= DEEPSEEK, messages= messages, quant= True, max_new_tokens= 1000) # Increasing new tokens limit as DEEPSEEK is a reasoning model

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<｜begin▁of▁sentence｜><｜User｜>Tell a joke for a room of Data Scientists.<｜Assistant｜><think>
Alright, so I need to come up with a joke for a room full of data scientists. The key here is to make it something that's funny but also relevant to their work. They're all experts in data, so the joke should be something that could potentially make them laugh or at least be a bit amusing.

First, I should think about what data scientists do. They work with numbers, data, statistics, and often have to deal with messy or incomplete datasets. They might be working on predictive models, data visualization, or statistical analysis. So, the joke should probably relate to these areas.

I want the joke to be light-hearted but not too silly. It should probably play on a common phrase or a scenario that's relatable. Maybe something that involves a common object or activity that data scientists might be involved in.

Let me think about a common scenario: working with data. Maybe something like analyzing d

**LOL!! Not for the actual joke which I didn't find funny, but for all the overthinking, which I can relate with! :D**

`Don't forget to delete downloaded models from C:\Users\shail\.cache\huggingface\hub\`